In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import sklearn.model_selection 
from sklearn.linear_model import LogisticRegression

In [2]:
data = pd.read_csv('heart.csv')
data.shape

(303, 14)

In [3]:
# a)
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(data.iloc[:,: -1], data.target,  test_size = 0.2,
                                                                            random_state = 0)

In [4]:
regression = LogisticRegression(max_iter = 1000)
regression.fit(X_train, y_train)
pred = regression.predict(X_test)
sklearn.metrics.accuracy_score(y_test, pred)

0.8524590163934426

In [4]:
a = data.iloc[:,:-1]

In [6]:
mew = []
for j in range(a.shape[1]):
    m = 0
    for i in range(a.shape[0]):
        m += a.iloc[i, j]
    mew.append(m / a.shape[0])
mew

[54.366336633663366,
 0.6831683168316832,
 0.966996699669967,
 131.62376237623764,
 246.26402640264027,
 0.1485148514851485,
 0.528052805280528,
 149.64686468646866,
 0.32673267326732675,
 1.0396039603960396,
 1.3993399339933994,
 0.7293729372937293,
 2.3135313531353137]

In [7]:
sig = []
for j in range(a.shape[1]):
    g = 0
    for i in range(a.shape[0]):
        g += (a.iloc[i, j] - mew[j]) ** 2
    sig.append(np.sqrt(g / a.shape[0]))
    
sig

[9.067101638577872,
 0.46524119304834705,
 1.0303480250839467,
 17.509178065734385,
 51.74515101045714,
 0.3556096038825336,
 0.5249911240963213,
 22.867332581889233,
 0.4690185854386941,
 1.1591574732421361,
 0.6152084301256662,
 1.0209175011165654,
 0.6112653149988241]

In [8]:
#My implementaion for data normalization
norm = np.ones(a.shape)
for i in range(a.shape[0]):
    for j in range(a.shape[1]):
        norm[i, j] = (a.iloc[i, j] - mew[j]) / sig[j]

norm = pd.DataFrame(norm, columns=['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal'])

In [9]:
from sklearn.preprocessing import StandardScaler

In [5]:
#Sklearn's implementation
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled = scaler.fit_transform(a)
scaled = pd.DataFrame(scaled, columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal'])

In [6]:
#Doing logisticResgrssion after data normalization
#We can see that the results are same; however, with data norm. alogrithm works with max_depth 12 which is much less then
# without normalization
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(scaled, data.target,  test_size = 0.2,
                                                                            random_state = 0)
regression = LogisticRegression(max_iter = 12)
regression.fit(X_train, y_train)
pred = regression.predict(X_test)
sklearn.metrics.accuracy_score(y_test, pred)

0.8524590163934426

In [12]:
#without norm 
import sklearn.neural_network
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(data.iloc[:,: -1], data.target,  test_size = 0.2, 
                                                                            random_state = 0)
model = sklearn.neural_network.MLPClassifier(hidden_layer_sizes=2, activation='relu',max_iter=500, solver='adam',
                                            alpha= 0.15, batch_size=20)
model.fit(X_train, y_train)
model.predict(X_test)
sklearn.metrics.accuracy_score(y_test, pred)

0.8524590163934426

In [17]:
#with norm 
import sklearn.neural_network
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(norm, data.target,  test_size = 0.2,
                                                                            random_state = 0)
model = sklearn.neural_network.MLPClassifier(hidden_layer_sizes=2, activation='relu', max_iter=500, solver='adam',
                                            alpha= 0.15, batch_size=20)
model.fit(X_train, y_train)
model.predict(X_test)
sklearn.metrics.accuracy_score(y_test, pred)

0.8524590163934426

In [47]:
from tensorflow.keras import regularizers
from tensorflow.keras.layers import BatchNormalization

In [73]:
#With tensorflow
opt = tf.keras.optimizers.Adam(learning_rate=0.0015)

model = tf.keras.Sequential([
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(150, activation = 'relu', kernel_regularizer = tf.keras.regularizers.l2(0.02)),
    tf.keras.layers.Dense(100, activation = 'relu', kernel_regularizer = tf.keras.regularizers.l2(0.01)),
    tf.keras.layers.Dense(2, activation = 'sigmoid')
    
    
])

model.compile(optimizer= opt, 
              loss = 'sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(X_train , y_train, epochs = 30)

Epoch 1/30
8/8 [==============================] - 1s 2ms/step - loss: 2.2192 - accuracy: 0.6529
Epoch 2/30
8/8 [==============================] - 0s 2ms/step - loss: 1.8838 - accuracy: 0.8099
Epoch 3/30
8/8 [==============================] - 0s 2ms/step - loss: 1.6549 - accuracy: 0.8347
Epoch 4/30
8/8 [==============================] - 0s 2ms/step - loss: 1.4886 - accuracy: 0.8058
Epoch 5/30
8/8 [==============================] - 0s 2ms/step - loss: 1.2810 - accuracy: 0.8182
Epoch 6/30
8/8 [==============================] - 0s 2ms/step - loss: 1.1861 - accuracy: 0.8140
Epoch 7/30
8/8 [==============================] - 0s 2ms/step - loss: 1.0591 - accuracy: 0.8471
Epoch 8/30
8/8 [==============================] - 0s 2ms/step - loss: 1.0082 - accuracy: 0.8099
Epoch 9/30
8/8 [==============================] - 0s 2ms/step - loss: 0.9178 - accuracy: 0.8140
Epoch 10/30
8/8 [==============================] - 0s 2ms/step - loss: 0.8368 - accuracy: 0.8223
Epoch 11/30
8/8 [======================

In [74]:
model.evaluate(X_test, y_test)

2/2 [==============================] - 0s 2ms/step - loss: 0.4286 - accuracy: 0.8852


[0.42863914370536804, 0.8852459192276001]

In [ ]:
model.we